<div style="background:linear-gradient(135deg,#1b1448 0%,#3b1f5e 55%,#7c2d12 100%);border-radius:18px;padding:34px 30px;color:#fff;font-family:Inter,Segoe UI,sans-serif">
  <div style="font-size:12px;letter-spacing:3px;color:#fed7aa;font-weight:700;text-transform:uppercase">Chapter 1 · Solutions</div>
  <div style="font-size:34px;font-weight:900;line-height:1.1;margin:10px 0 6px">What building AI with AI means · Solutions</div>
  <div style="font-size:15px;color:#ffe4d6;max-width:740px;line-height:1.6">Worked answers to the chapter's three practice challenges, replayed from recorded calls.</div>
  <div style="font-size:13px;color:#fed7aa;margin-top:14px">Building AI with AI: A Visual Handbook · John Fisher</div>
</div>

Worked answers to the three practice challenges in Chapter 1, What building AI with AI means. Each answer replays calls recorded for the book with Claude Sonnet 5.5 on November 10, 2026, and every ticket in it is written for the book. Try each challenge in the chapter's notebook before reading its answer here.

## Setup

The same Setup as the chapter's notebook, with this notebook's own recording.

In [1]:
# Set LIVE = True to send these requests to the Anthropic API with your own key, saved in
# Colab's secrets panel as ANTHROPIC_API_KEY. Left False, the notebook replays the answers
# recorded for the book, with no key and no cost.
LIVE = False
%pip install -q --disable-pip-version-check anthropic==2.7.0
import importlib, pathlib, sys, urllib.request
if "google.colab" in sys.modules:
    BASE = "https://raw.githubusercontent.com/johnfisher-ai/ai-with-ai/main/chapters/notebooks/"
    pathlib.Path("recordings").mkdir(exist_ok=True)
    for name in ["book_client.py", "recordings/what-building-ai-with-ai-means--solutions.json"]:
        urllib.request.urlretrieve(BASE + name, name)
import book_client
importlib.reload(book_client)
client = book_client.make_client("what-building-ai-with-ai-means--solutions", mode="live" if LIVE else None)

Note: you may need to restart the kernel to use updated packages.


Replaying 6 recorded responses from claude-sonnet-5-5, recorded 2026-11-10. Set LIVE = True in Setup to call the API yourself.


In [2]:
# The tickets in these answers are written for this book, like the chapter's. No customer wrote them.
MODEL = "claude-sonnet-5-5"


def answer_text(reply):
    """The text of a reply. Its content is a list of blocks, and the text blocks hold the answer."""
    return "".join(block.text for block in reply.content if block.type == "text")


def ask(system, text, max_tokens=1024):
    reply = client.messages.create(model=MODEL, max_tokens=max_tokens, system=system,
                                   messages=[{"role": "user", "content": text}])
    return answer_text(reply).strip()

## Challenge 1. Add a returns category

Add `returns` to the list of categories, then sort two tickets about returns and the crushed box from the chapter.

In [3]:
# Challenge 1. A fifth category, returns, and three tickets that test it.
CATEGORIES = ["billing", "login", "shipping", "returns", "other"]
SYSTEM = ("You sort customer support tickets for an online store. "
          "Reply with exactly one word from this list: " + ", ".join(CATEGORIES) + ".")
for ticket in ["I'd like to send back a sweater that doesn't fit.",
               "How long does a refund take once you receive my return?",
               "The box arrived crushed and two of the mugs inside are broken."]:
    label = ask(SYSTEM, ticket).lower()
    print(f"{label if label in CATEGORIES else 'unrecognized':<13} {ticket}")

returns       I'd like to send back a sweater that doesn't fit.
returns       How long does a refund take once you receive my return?
shipping      The box arrived crushed and two of the mugs inside are broken.


Both return tickets went to the new `returns` category, including the one that asks about a refund, which could have gone to `billing`. The crushed box stayed in `shipping`. Three tickets can't show that the new category left every other label alone, and Part 3 checks a change like this one against a whole test set.

## Challenge 2. Ask for a reason

Change the system prompt so the reply is a category and a one-sentence reason, then split the two apart. Try it on the two tickets the chapter called judgment calls.

In [4]:
# Challenge 2. Ask for the category and a reason, on the two tickets that could fit more than one.
CATEGORIES = ["billing", "login", "shipping", "other"]
SYSTEM_WITH_REASON = ("You sort customer support tickets for an online store. Reply on one line as "
                      "category: reason, where category is exactly one word from this list: "
                      + ", ".join(CATEGORIES) + ", and the reason is one short sentence.")
for ticket in ["The box arrived crushed and two of the mugs inside are broken.",
               "I can't log in to download the invoice for my last order."]:
    category, _, reason = ask(SYSTEM_WITH_REASON, ticket).partition(":")
    print(ticket)
    print(f"  category: {category.strip().lower()}")
    print(f"  reason:   {reason.strip()}")

The box arrived crushed and two of the mugs inside are broken.
  category: shipping
  reason:   The order was damaged in transit, so the package arrived crushed with broken items.
I can't log in to download the invoice for my last order.
  category: login
  reason:   The customer cannot access their account, which blocks them from downloading the invoice.


The labels match the chapter's, `shipping` and `login`, and each reason says why. For the box, the model wrote "The order was damaged in transit, so the package arrived crushed with broken items." A reason makes a label easier to review. It also adds output tokens to every call, and the format can break, so code that splits on the colon should expect a reply without one.

## Challenge 3. Write three more cases

Send the chapter's request again, then test the function that comes back with cases of your own. These three are shouting with extra punctuation, a question mark, and a reply that names two categories.

In [5]:
REQUEST = """Write a Python function normalize_label(raw: str) -> str.
It takes a model's one-word reply, such as " Billing.\n", and returns a clean category.
Lowercase it, strip whitespace and any trailing punctuation, and return the result if it is one of
billing, login, shipping or other. Return "other" for anything else.
Reply with only the code, in one Python code block, with no explanation."""

# Challenge 3. The same request as the notebook, then three more cases of your own.
reply = client.messages.create(model=MODEL, max_tokens=2048,
                               messages=[{"role": "user", "content": REQUEST}])
written = answer_text(reply)
print(written)

import re
match = re.search(r"```(?:python)?\n(.*?)```", written, re.S)
namespace = {}
exec(match.group(1) if match else written, namespace)
normalize_label = namespace["normalize_label"]

MORE_CASES = [("  SHIPPING!!", "shipping"), ("Login?", "login"), ("billing and login", "other")]
for raw, expected in MORE_CASES:
    got = normalize_label(raw)
    print(f"{'pass' if got == expected else 'FAIL'}  normalize_label({raw!r}) -> {got!r}, expected {expected!r}")

```python
import string

VALID_LABELS = {"billing", "login", "shipping", "other"}


def normalize_label(raw: str) -> str:
    if not isinstance(raw, str):
        return "other"
    cleaned = raw.strip().lower()
    cleaned = cleaned.strip(string.whitespace + string.punctuation)
    return cleaned if cleaned in VALID_LABELS else "other"
```
pass  normalize_label('  SHIPPING!!') -> 'shipping', expected 'shipping'
pass  normalize_label('Login?') -> 'login', expected 'login'
pass  normalize_label('billing and login') -> 'other', expected 'other'


All three cases pass. The request was the same as the chapter's, but the function that came back differs from the chapter's in two small ways. It has fewer blank lines, and it lists whitespace before punctuation in the call that strips them, which changes nothing about what it does. Two calls with the same request returned different code, and the same cases would have caught a difference that mattered.

---
<div style="text-align:center;color:#666f8e;font-size:12px;margin-top:10px">Building AI with AI: A Visual Handbook · © 2026 John Fisher · text CC BY 4.0, code MIT</div>